In [19]:
import sqlite3
import time
import requests
import asyncio
import nest_asyncio
from bs4 import BeautifulSoup
import re
from playwright.async_api import async_playwright

# Allow nested event loops in Jupyter
nest_asyncio.apply()

DB_PATH = "courses.db"

def get_term_tables(cursor):
    """Finds all tables that look like 'courses_202610'."""
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'courses_%'")
    # Extract the table names and ignore the global_search table if it doesn't match the pattern
    return [row[0] for row in cursor.fetchall()]

def ensure_eval_column(cursor, table_name):
    """Ensures the eval_score column exists in the given table."""
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = [col[1] for col in cursor.fetchall()]
    
    if "eval_score" not in columns:
        print(f"Adding 'eval_score' column to {table_name}...")
        cursor.execute(f"ALTER TABLE {table_name} ADD COLUMN eval_score TEXT")

async def get_human_cookies():
    """Use Playwright to let the user log in, then steal the session cookies."""
    async with async_playwright() as p:
        print("Launching browser... Please log in and pass Duo!")
        browser = await p.chromium.launch(headless=False) 
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto("https://esther.rice.edu/")

        # Wait indefinitely for the user to log in and the main menu to appear
        await page.wait_for_selector("text='Personal Information'", timeout=0)
        print("✅ Logged in successfully! Snatching cookies...\n")
        
        playwright_cookies = await context.cookies()
        await browser.close()
        
        return {cookie['name']: cookie['value'] for cookie in playwright_cookies}

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [20]:
cookies = asyncio.run(get_human_cookies())


# 3. Setup Requests
session = requests.Session()
requests.utils.add_dict_to_cookiejar(session.cookies, cookies)
url = "https://esther.rice.edu/selfserve/swkscmt.main"
headers = {
    "Content-Type": "application/x-www-form-urlencoded",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

Launching browser... Please log in and pass Duo!
✅ Logged in successfully! Snatching cookies...



In [21]:


# print("🚀 Starting the scraping engine. Grab a coffee...\n")
# success_count = 0
# fail_count = 0

# # 4. Scrape Loop
# for course in all_courses_to_scrape:
payload = {
    "p_commentid": "",
    "p_confirm": "1",
    "p_term": '202610',
    "p_type": "Course",
    # "p_subj": 'ELEC',
"p_crn": '12423'
}


response = session.post(url, headers=headers, data=payload)

response.text

#     # Session timeout check
if "bmenu.P_MainMnu" not in response.text and "Personal Information" not in response.text:
    print("\n❌ Session expired or blocked. Stopping script.")


soup = BeautifulSoup(response.text, 'html.parser')
# soup.prettify()
response.text
#     mean_divs = soup.find_all('div', class_='third', string=re.compile(r'Class Mean:'))
    
#     if not mean_divs:
#         score_to_save = "N/A"
#     else:
#         try:
#             # Grab the 3rd chart's mean (usually "Overall Quality")
#             if len(mean_divs) >= 3:
#                 score_to_save = mean_divs[2].text.replace("Class Mean: ", "").strip()
#             else:
#                 score_to_save = mean_divs[0].text.replace("Class Mean: ", "").strip()
#         except Exception:
#             score_to_save = "Error"

#     # 5. Save back to the SPECIFIC term table this course belongs to
#     cursor.execute(
#         f"UPDATE {course['table']} SET eval_score = ? WHERE crn = ?", 
#         (score_to_save, course["crn"])
#     )
#     conn.commit()

#     if score_to_save not in ["N/A", "Error"]:
#         print(f"✅ {course['term']} | {course['crs']} (CRN {course['crn']}): {score_to_save}")
#         success_count += 1
#     else:
#         print(f"⚠️ {course['term']} | {course['crs']} (CRN {course['crn']}): No data")
#         fail_count += 1

#     time.sleep(0.5)

# print(f"\n🏁 Finished! Successfully scraped {success_count} scores. ({fail_count} had no data).")

# # Optional: If your global_search table is a real table and not just a VIEW, 
# # you might want to rebuild it here after the script finishes updating the base tables!


'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/transitional.dtd">\n<HTML lang="en">\n<head>\n<META http-equiv="Content-Type" content="text/html; charset=UTF-8">\n<meta http-equiv="Pragma" name="Cache-Control" content="no-cache">\n<meta http-equiv="Cache-Control" name="Cache-Control" content="no-cache">\n<LINK REL="stylesheet" HREF="/css/web_defaultapp.css" TYPE="text/css">\n<LINK REL="stylesheet" HREF="/css/web_defaultprint.css" TYPE="text/css" media="print">\n<LINK REL="stylesheet" type="text/css" href="/css/custom.css">\n<LINK REL="stylesheet" type="text/css" href="/css/font-awesome.min.css">\n<meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=2, minimum-scale=1, user-scalable=yes">\n<title>Course and Instructor Evaluation Display</title>\n<meta http-equiv="Content-Script-Type" name="Default_Script_Language" content="text/javascript">\n<SCRIPT LANGUAGE="JavaScript" TYPE="text/javascript">\n<!-- Hide JavaScrip

In [30]:
results = soup.find('div', class_ = 'results-container')
charts = results.find('div', class_='charts')
comments = results.find('div', class_='comments')
charts

<div class="charts"><div class="title">Student Numeric Responses</div><div class="chart"><div class="filler"><div><div class="third">Class Mean: 2.33</div><div class="third">Rice Mean: 1.69</div><div class="third"></div></div><div><div class="third">Responses: 66</div><div class="third"></div><div class="third"></div></div><img src="/servlets/com.objectplanet.chart.ChartServlet?chart=bar&amp;barLabelsOn=true&amp;rangeLabelPostfix=%25&amp;valueLabelsOn=true&amp;valueLabelsOn=true&amp;valueLinesOn=true&amp;width=450&amp;height=200&amp;range=100&amp;sampleColors=%231C4777&amp;format=jpeg&amp;titleFont=Arial,Bold,11&amp;background=%23F5F6F7&amp;chartTitle= Organization: The course organization was:&amp;sampleValues=23,44,17,11,6&amp;sampleLabels=1%0AOutstanding,2%0AGood,3%0AAverage,4%0AFair,5%0APoor"/></div></div><div class="chart"><div class="filler"><div><div class="third">Class Mean: 2.32</div><div class="third">Rice Mean: 1.7</div><div class="third"></div></div><div><div class="third">

In [4]:
from calendar import c
import os

import bs4
import requests as r
from xml.etree import ElementTree as ET
import pandas as pd
from pandas import DataFrame
import sqlite3 as sql
import json

META_COURSES_URL = 'https://courses.rice.edu/courses/!SWKSCAT.info'
BASE_COURSES_URL = 'https://courses.rice.edu/'
BASE_GA_URL = 'https://ga.rice.edu'

courses = []
req = r.get(f"{BASE_COURSES_URL}/courses/courses/!SWKSCAT.cat?p_action=QUERY&p_term={'202610'}&p_school={'EN'}", timeout=15)
parser = bs4.BeautifulSoup(req.text, 'html.parser')
rows = parser.find('tbody').find_all('tr')
rows[0].find_all('td')

for row in parser.find('tbody').find_all('tr'):
    cells = row.find_all('td')
    if len(cells) != 7:
        raise ValueError(f"Expected 7 cells per row, got {len(cells)}")
    courses.append({
        'crn': cells[0].text,
        'crs': cells[1].text,
        'title': cells[3].text, #ignore "part of term" entry
        'instructors': json.dumps([instructor.text for instructor in cells[4].find_all('a')]),
        'meeting_times': json.dumps(list(cells[5].find('div', class_='mtg-clas').stripped_strings)), # ignore final exam time
        'credits': cells[6].text,
        'course_page': f"{BASE_COURSES_URL}{cells[0].a['href']}"
    })

In [ ]:
import api.index as index
index.cons